# Silver layer transformaion

### Decisions
- Dropped rows where `Event distance/length` contains `d` (days), as invalid per lab spec.
- Dropped rows where `Athlete performance` contains `d`, same reason.
- Dropped rows where event unit and performance unit match (km/km, mi/mi, h/h) because it indicates mismatched data.
- Rows where `Event distance/length` ends with `k` converted to `km` before filtering.
- `Athlete performance` converted to decimal hours (distance races) or decimal km (timed races).
- `event_id` created with sha2 on event name because dense_rank didn't work for streaming table.
- `athlete_id` created with sha2 on Athlete ID to enable a dim_athlete table in gold.

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col, when
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window
import hashlib


@dp.table(
    name="marathos.silver.races_clean",
    comment="Cleaned race data",
    table_properties={
        "delta.columnMapping.mode": "name",
        "delta.minReaderVersion": "2",
        "delta.minWriterVersion": "5",
    },
)

def races_clean():
    df = dp.read_stream("marathos.bronze.races")

    df_clean = (
        df
        .withColumn(
            "Event distance/length",
            F.regexp_replace(col("Event distance/length"), "k$", "km")
        )
        .filter(~col("Event distance/length").like("%d%"))
        .filter(~col("Athlete performance").like("%d%"))
        .filter(
            ~(
                col("Event distance/length").like("%km")
                & col("Athlete performance").like("%km")
            )
            & ~(
                col("Event distance/length").like("%mi")
                & col("Athlete performance").like("%mi")
            )
            & ~(
                col("Event distance/length").like("%h")
                & col("Athlete performance").like("%h")
            )
        )
        .withColumn(
            "Athlete performance", F.regexp_replace("Athlete performance", "h", "")
        )
        .withColumn("performance_split", F.split(F.col("Athlete performance"), ":"))
        .withColumn(
            "Athlete performance",
            when(
                col("Event distance/length").like("%h"),
                F.regexp_replace(col("Athlete performance"), " km", "").cast("double"),
            ).otherwise(
                F.col("performance_split")[0].cast("double")
                + F.col("performance_split")[1].cast("double") / 60
                + F.col("performance_split")[2].cast("double") / 3600
            ),
        )     
        .withColumn("event_id", F.sha2(col("Event name"), 256))
        .withColumn("athlete_id", F.sha2(col("Athlete ID").cast("string"), 256))
        .drop("performance_split")
    )

    return df_clean